# 05 - Demand patterns

Three views of the same curated games:

1. **Association rules** on binned hardware specs: which requirement combinations
   go with which demand tier.
2. **Demand over time**: required tier per 5-year release bucket, plus an OLS slope.
3. **Demand vs the GPU catalog**: each bucket's tier joined to `gpus.csv` to name the
   cheapest capable card, its percentile and coverage.

The GPU table has no release dates, so view 3 is relative to today's catalog.

Outputs: `analytics_rules.json`, `analytics_trend.csv`, `analytics_demand.json`.


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.fpm import FPGrowth
from pyspark.ml.regression import LinearRegression

JOBS = os.environ.get('JOBS_DIR', '/home/jovyan/jobs')
if JOBS not in sys.path:
    sys.path.insert(0, JOBS)
from schemas import GPUS_SCHEMA
from catalog import TIER_BOUNDS, band
from artifacts import write_json, write_rows_csv

from spark_session import get_spark

DATA = Path(os.environ.get('DATA_DIR', '/home/jovyan/data'))
spark = get_spark('gpu-recsys-demand')
print('spark', spark.version)

In [ ]:
features = spark.read.parquet(str(DATA / 'curated' / 'features.parquet')).cache()
print(f'analytics frame: {features.count():,} games')


RAM = [((F.col('min_ram_gb') > 0) & (F.col('min_ram_gb') <= 4), 'RAM 0-4'),
       ((F.col('min_ram_gb') > 4) & (F.col('min_ram_gb') <= 8), 'RAM 5-8'),
       ((F.col('min_ram_gb') > 8) & (F.col('min_ram_gb') <= 16), 'RAM 9-16'),
       (F.col('min_ram_gb') > 16, 'RAM 17+'),
       (F.col('min_ram_gb') <= 0, 'RAM unknown')]
VRAM = [((F.col('min_vram_gb') > 0) & (F.col('min_vram_gb') <= 2), 'VRAM 0-2'),
        ((F.col('min_vram_gb') > 2) & (F.col('min_vram_gb') <= 4), 'VRAM 3-4'),
        ((F.col('min_vram_gb') > 4) & (F.col('min_vram_gb') <= 8), 'VRAM 6-8'),
        (F.col('min_vram_gb') > 8, 'VRAM 9+'),
        (F.col('min_vram_gb') <= 0, 'VRAM unknown')]
GHZ = [((F.col('min_ghz') > 0) & (F.col('min_ghz') < 2.5), 'Clock <2.5'),
       ((F.col('min_ghz') >= 2.5) & (F.col('min_ghz') <= 3.2), 'Clock 2.5-3.2'),
       ((F.col('min_ghz') > 3.2) & (F.col('min_ghz') <= 4.0), 'Clock 3.2-4'),
       (F.col('min_ghz') > 4.0, 'Clock >4'),
       (F.col('min_ghz') <= 0, 'Clock unknown')]
TIER = [(F.col('required_g3d') <= TIER_BOUNDS[0], 'Tier light'),
        (F.col('required_g3d') <= TIER_BOUNDS[1], 'Tier entry'),
        (F.col('required_g3d') <= TIER_BOUNDS[2], 'Tier mid'),
        (F.col('required_g3d') <= TIER_BOUNDS[3], 'Tier high'),
        (F.col('required_g3d') > TIER_BOUNDS[3], 'Tier extreme')]
AGE = [(F.col('game_age') <= 5, 'Age 0-5'), (F.col('game_age') <= 10, 'Age 6-10'),
       (F.col('game_age') <= 20, 'Age 11-20'), (F.col('game_age') > 20, 'Age 21+')]

items_df = features.withColumn(
    'items',
    F.array(
        band('min_ram_gb', RAM), band('min_vram_gb', VRAM), band('min_ghz', GHZ),
        band('required_g3d', TIER), band('game_age', AGE),
        F.when(F.col('is_capped') == 1, 'Cap on').otherwise('Cap off'),
        F.when(F.coalesce(F.col('fps_1080p_high_iris_xe'), F.lit(-1)) >= 60, 'Integrated ok').otherwise('Integrated slow'),
    ),
).select('name', 'release_year', 'required_g3d', 'game_age', 'items')
items_df.show(3, truncate=False)

## Association rules

FPGrowth needs support >= 1% and confidence >= 0.6. Trivial rules are dropped: at
least two conditions, and the outcome must be a demand tier. Rules are ranked by
lift (lift > 1 beats chance).


In [ ]:
growth = FPGrowth(itemsCol='items', minSupport=0.01, minConfidence=0.6).fit(items_df)
n_itemsets = growth.freqItemsets.count()
n_rules = growth.associationRules.count()
print(f'frequent itemsets: {n_itemsets:,} | rules: {n_rules:,}')

rules = (
    growth.associationRules
    .filter(F.size('antecedent') >= 2)
    .filter(F.concat_ws(',', F.col('consequent')).rlike('Tier '))
    .orderBy(F.desc('lift'))
    .limit(40)
    .collect()
)
print(f'{"antecedent":<46} {"consequent":<16} {"support":>8} {"conf":>6} {"lift":>7}')
for rule in rules:
    print(f'{", ".join(rule["antecedent"]):<46} {", ".join(rule["consequent"]):<16} '
          f'{rule["support"]:>8.3f} {rule["confidence"]:>6.3f} {rule["lift"]:>7.2f}')

## Demand over time

5-year buckets. Two curves: the median minimum GPU score and the median recommended
GPU score from the specs. The recommended field is missing for many old titles, so
that curve is thinner.


In [ ]:
years_df = (
    features
    .filter(F.col('release_year').isNotNull())
    .withColumn('bucket', (F.floor(F.col('release_year') / 5) * 5).cast('int'))
    .groupBy('bucket')
    .agg(F.count('*').alias('games'),
         F.percentile_approx('required_g3d', 0.5).alias('median_tier'),
         F.percentile_approx('required_g3d', 0.9).alias('p90_tier'),
         F.percentile_approx('min_gpu_g3d', 0.5).alias('median_min_score'),
         F.percentile_approx('rec_gpu_g3d', 0.5).alias('median_rec_score'),
         F.sum(F.col('rec_gpu_g3d').isNotNull().cast('int')).alias('rec_n'),
         F.round(F.avg(((F.coalesce(F.col('fps_1080p_high_iris_xe'), F.lit(-1)) >= 60)
                        | (F.coalesce(F.col('fps_1080p_high_radeon_780m'), F.lit(-1)) >= 60)).cast('int')), 3).alias('share_igpu_ok'),
         F.round(F.avg('is_capped'), 3).alias('share_capped'),
         F.round(F.avg((F.col('required_g3d') >= 17009).cast('int')), 3).alias('share_high_tier'))
)
years = years_df.orderBy('bucket').collect()
def show(value):
    return '-' if value is None else value

print(f'{"period":<10}{"games":>7}{"median":>8}{"min":>8}{"rec":>8}{"rec_n":>6}{"igpu_ok":>9}{"capped":>8}{"high":>7}')
for row in years:
    print(f'{str(row["bucket"]) + "-" + str(row["bucket"] + 4):<10}{row["games"]:>7}{row["median_tier"]:>8}'
          f'{show(row["median_min_score"]):>8}{show(row["median_rec_score"]):>8}{row["rec_n"]:>6}'
          f'{row["share_igpu_ok"]:>9}{row["share_capped"]:>8}{row["share_high_tier"]:>7}')

## Demand vs the GPU catalog

For each bucket's median and p90 tier: the cheapest card that meets it, where the
tier sits in the 1,381-card catalog, and the share of cards that meet it. The GPU
table has no release years, so this is today's catalog.


In [ ]:
cards = spark.read.csv(str(DATA / 'gpus.csv'), header=True, schema=GPUS_SCHEMA).select('name', 'g3d_mark')

tiers_long = (
    years_df.select('bucket', F.col('median_tier').alias('tier'))
    .union(years_df.select('bucket', F.col('p90_tier').alias('tier')))
    .distinct()
)

stats = (
    tiers_long.crossJoin(cards)
    .groupBy('bucket', 'tier')
    .agg(F.count('*').alias('n_cards'),
         F.sum((F.col('g3d_mark') >= F.col('tier')).cast('int')).alias('meet'),
         F.min(F.when(F.col('g3d_mark') >= F.col('tier'), F.col('g3d_mark'))).alias('cheapest_score'))
    .withColumn('percentile', F.round((F.col('n_cards') - F.col('meet')) / F.col('n_cards'), 4))
    .withColumn('coverage', F.round(F.col('meet') / F.col('n_cards'), 4))
)

name_by_score = cards.groupBy(F.col('g3d_mark').alias('cheapest_score')).agg(F.min('name').alias('cheapest_card'))
catalog_rows = stats.join(name_by_score, on='cheapest_score', how='left').orderBy('bucket', 'tier').collect()

gpu_info = {}
print(f'{"bucket":>6} {"tier":>6} {"card":<30} {"card_g3d":>8} {"pctile":>7} {"coverage":>8}')
for row in catalog_rows:
    gpu_info[(row['bucket'], row['tier'])] = {
        'cheapest_card': row['cheapest_card'],
        'cheapest_score': row['cheapest_score'],
        'percentile': row['percentile'],
        'coverage': row['coverage'],
    }
    print(f'{row["bucket"]:>6} {row["tier"]:>6} {str(row["cheapest_card"])[:30]:<30} '
          f'{row["cheapest_score"]:>8} {row["percentile"]:>7} {row["coverage"]:>8}')

missing = [k for k, v in gpu_info.items() if v['cheapest_card'] is None or v['cheapest_score'] < k[1]]
print('sanity (no missing card, cheapest >= tier):', 'PASS' if not missing else f'FAIL {missing}')

In [ ]:
vw = VectorAssembler(inputCols=['release_year'], outputCol='year_v')
trend_df = vw.transform(features.withColumn('req_log', F.log(F.col('required_g3d'))))
lr = LinearRegression(featuresCol='year_v', labelCol='req_log', maxIter=50).fit(trend_df)
slope = float(lr.coefficients[0])
intercept = float(lr.intercept)
print(f'OLS on log(required_g3d): slope={slope:.5f} per year, intercept={intercept:.3f}, r2={lr.summary.r2:.3f}')
print(f'tier multiplier over 10 years: e^(10*slope) = {pow(2.718281828, slope * 10):.3f}x')

rules_path = DATA / 'curated' / 'analytics_rules.json'
write_json(rules_path, {
    'params': {'minSupport': 0.01, 'minConfidence': 0.6, 'min_antecedent': 2, 'consequent': 'Tier *'},
    'frequent_itemsets': n_itemsets,
    'rules_total': n_rules,
    'top_rules': [{'antecedent': r['antecedent'], 'consequent': r['consequent'],
                   'support': round(r['support'], 4), 'confidence': round(r['confidence'], 4),
                   'lift': round(r['lift'], 3)} for r in rules],
})

def bucket_row(row):
    med = gpu_info.get((row['bucket'], row['median_tier']), {})
    p90 = gpu_info.get((row['bucket'], row['p90_tier']), {})
    return {
        'period_start': row['bucket'],
        'games': row['games'],
        'median_tier': row['median_tier'],
        'median_tier_card': med.get('cheapest_card'),
        'median_tier_percentile': med.get('percentile'),
        'catalog_share_meeting_median': med.get('coverage'),
        'p90_tier': row['p90_tier'],
        'median_min_g3d': row['median_min_score'],
        'median_rec_g3d': row['median_rec_score'],
        'rec_n': row['rec_n'],
        'p90_tier_card': p90.get('cheapest_card'),
        'p90_tier_percentile': p90.get('percentile'),
        'catalog_share_meeting_p90': p90.get('coverage'),
        'share_igpu_ok': row['share_igpu_ok'],
        'share_capped': row['share_capped'],
        'share_high_tier': row['share_high_tier'],
    }

trend_rows = [bucket_row(row) for row in years]
trend_csv = DATA / 'curated' / 'analytics_trend.csv'
write_rows_csv(trend_csv, list(trend_rows[0].keys()), trend_rows)

n_cards = cards.count()
demand_path = DATA / 'curated' / 'analytics_demand.json'
write_json(demand_path, {
    'ols': {'label': 'log(required_g3d)', 'slope_per_year': round(slope, 6),
            'intercept': round(intercept, 4), 'r2': round(float(lr.summary.r2), 4),
            'multiplier_over_10y': round(pow(2.718281828, slope * 10), 3)},
    'catalog': {'cards': n_cards, 'basis': 'current snapshot; gpus.csv has no release years'},
    'buckets': trend_rows,
})

## Caveats

- No GPU release dates: the trend is game demand, and catalog columns are today's.
- More titles are sampled in recent years; read counts with the medians.
- `required_g3d` is one of 12 reference tiers, so medians step; the min/rec curves
  come straight from the specced scores.
- 2026/2027 rows are future-dated in the source.
- Engine-capped games are excluded.
- "Cheapest" is the lowest score that meets the tier; ties show the first name.
